# 13. 데이터 검증, Pydantic

Pydantic **BaseModel**로 데이터의 모양을 선언하고, 자동 검증,형변환,JSON변환을 활용하여, 챗봇의 요청,응답 모델 (app/models.py)을 만든다.

---

**챗봇 역할** 서버로 들어오는 요청, 나가는 응답의 모양을 정한다. 

# 1. 왜 Pydantic을 써야 하는가?

```python
if role not in ALLOWED_ROLES:
    raise ValueError(...)
if not content.strip():
    raise ValueError(...)
```

검사 항목이 들어날수록 **if**문이 많아지고, Type, Length, 필수 값, 중첩 데이터 등을 다 조건문으로 확인해야한다.

<br>

**Pydantic**은 Type Hint로 데이터의 모양만 선언하면, 실행중에 실제로 직접 검사해준다.

## 2. 첫 모델 - **BaseModel**

In [2]:
from typing import Literal  
from pydantic import BaseModel, ValidationError

class Message(BaseModel):
    role : Literal["system", "user", "assistant"]
    content: str

msg = Message(role="user", content="연차는 며칠이야?")
print(msg)
print(msg.role)
print(msg.model_dump())
print(msg.model_dump_json)

role='user' content='연차는 며칠이야?'
user
{'role': 'user', 'content': '연차는 며칠이야?'}
<bound method BaseModel.model_dump_json of Message(role='user', content='연차는 며칠이야?')>


| 메서드 |	변환 |	쓰임 |
|---|---|---|
|**model_dump()**	|모델 → 딕셔너리|	LLM API에 넘길 때, 파일에 저장할 때|
|**model_dump_json()**	|모델 → JSON 문자열|	네트워크로 보낼 때|
|**모델.model_validate(dict)**	|딕셔너리 → 모델 (검증)|	파일에서 읽은 데이터를 확인할 때|
|**모델.model_validate_json(str)**	|JSON 문자열 → 모델 (검증)|	요청 본문, LLM의 JSON 답변을 확인할 때|

Literal은 변수, 함수 인자, 반환값 등이 지정된 특정 값만 가질 수 있도록 타입 체커에 알려줍니다.

In [7]:
test = Message(role="system", content="당신은 유용한 비서입니다.")
try:
    test2 = Message(role="king", content=123)
except ValidationError as e:
    print(e)
try:
    test3 = Message(role="int", content="당신은 유용한 비서입니다.")
except ValidationError as e:
    print(e)
try:
    test3 = Message(role=True, content=123)
except ValidationError as e:
    print(e)

2 validation errors for Message
role
  Input should be 'system', 'user' or 'assistant' [type=literal_error, input_value='king', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
content
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/string_type
1 validation error for Message
role
  Input should be 'system', 'user' or 'assistant' [type=literal_error, input_value='int', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
2 validation errors for Message
role
  Input should be 'system', 'user' or 'assistant' [type=literal_error, input_value=True, input_type=bool]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
content
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydant

## 4. 자동 형변환

LLM 설정값을 모델로 만들어본다. **.env**의 값은 항상 문자열이라 직접 **int()** 로 바꿔야 했다.

In [4]:
from pydantic import BaseModel, ConfigDict, ValidationError

class Settings(BaseModel):
    model_config = ConfigDict(frozen=True)

    model: str = "gpt-6-luna"
    max_tokens: int = 500
    temperature: float = 0.7
    stream: bool = False

# default의 d
d = Settings()
print(f"Setting 전 : ");print(d); print(type(d.max_tokens)) 
print("-"*50)

s = Settings(max_tokens="1000", temperature="0.2", stream="true")
print(s)
print(type(s.max_tokens))

print('-'*30)
try:
    Settings(max_tokens="많이")
except ValidationError as e:
    print(e)

Setting 전 : 
model='gpt-6-luna' max_tokens=500 temperature=0.7 stream=False
<class 'int'>
--------------------------------------------------
model='gpt-6-luna' max_tokens=1000 temperature=0.2 stream=True
<class 'int'>
------------------------------
1 validation error for Settings
max_tokens
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='많이', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_parsing


In [5]:
s.max_tokens = "hello"
print(s)

ValidationError: 1 validation error for Settings
max_tokens
  Instance is frozen [type=frozen_instance, input_value='hello', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/frozen_instance

### 왜 이런 차이가 날까?
- **정적 타입 검사기(에디터 빨간줄):** 코드를 실행하지 않고 Type Hint만 분석합니다. max_tokens가 int로 선언되어 있으므로, String 리터럴 "1000"을 넘기면 Type 불일치 경고를 냅니다.

- **Pydantic 실행기(실제 출력):** 코드가 실행될 때 Pydantic의 **타입 강제 변환(type coercion)** 기능이 동작해 "1000"을 1000(int), "0.2"를 0.2(float), "true"를 True(bool)로 바꿔 줍니다.

## 5. 세부 조건 - **Field**와 **field_validator**

### 5-1 **Field**로 길이 제한

너무 긴 질문은 LLM 비용을 크게 늘린다. 빈 질문은 보낼 필요가 없다.

In [8]:
from pydantic import Field

class Message(BaseModel):
    role : Literal["system", "user", "assistant"]
    content: str = Field(min_length=1, max_length=2000)

### 5-2. **field_validator**로 직접 규칙 추가

공백만 있는 메시지 **"    "**는 길이가 4라서 **min_length=1**을 통과한다. 앞뒤 공백을 지우고 검사하는 규칙을 **strip()** 로 추가한다

In [9]:
from pydantic import field_validator

class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1, max_length=2000)

    @field_validator("content")
    @classmethod
    def strip_content(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("내용이 비어 있다")
        return value

ImportError: attempted relative import with no known parent package